# 4. Feature engineering: from a request to ranking features

Run the three representation notebooks first. This notebook turns an impression and its candidate articles into model inputs using **information available before that impression’s feedback**.

Read it in four stages:
1. **Model contract:** understand which features enter a ranker and which fields must stay outside it.
2. **Worked methods:** follow small examples beside the actual profile, freshness, feature and sampling code.
3. **Dataset build:** apply those same methods to the saved MIND inputs, or reuse a verified existing run.
4. **Audit and handoff:** inspect coverage, values, sampling, runtime and inference parity.

The teaching examples are synthetic, deliberately small, and never written into the project’s datasets. The final audit uses your actual saved data. All displayed results are produced by executable cells.

## 0. Setup and run controls

Core calculations remain visible here. Imported helpers handle file/cache validation, keyed input checks, storage and reports. You do not run those helper files separately.

`REBUILD=False` reuses saved compatible outputs. After editing a calculation, use `REBUILD=True` and rebuild dependent datasets. The examples always execute, even when the dataset itself is loaded from cache.

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next(
    (
        p
        for p in (
            Path.cwd(),
            *Path.cwd().parents,
            Path("/content/drive/MyDrive/BT4222"),
        )
        if (p / "src/newsrec").is_dir()
    ),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the project checkout containing src/newsrec/."
    )
sys.path.insert(0, str(CODE_ROOT / "src"))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = (
    None  # Optional: point to another folder holding the project's saved datasets.
)
ROOT = (
    Path(DATA_ROOT or os.environ.get("NEWSREC_ROOT") or CODE_ROOT)
    .expanduser()
    .resolve()
)
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data/output root:", ROOT)

from newsrec.io import feature_run, feature_sample
import hashlib
from collections import deque
from newsrec.common import DEFAULT_SETTINGS
from newsrec.temporal import TemporalState as _StateStorage, User, HOUR
from newsrec.common import prepare_feature_pairs, validate_feature_schema
from newsrec.representations import Bundle
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.common import normalise
from dataclasses import replace
from types import SimpleNamespace

In [ ]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

display(
    pd.DataFrame(
        {
            "setting": [
                "history_cap",  # Maximum number of historical items to consider
                "recent_count",  # Number of recent items to consider
                "recent_hours",  # Time window in hours for recent items
                "interest_count",  # Number of interest items to consider
                "negative_count",  # Number of negative items to consider
                "context_features",  # List of context features to use
                "update_mode",
            ],
            "value": [
                SETTINGS.history_cap,
                SETTINGS.recent_count,
                SETTINGS.recent_hours,
                SETTINGS.interest_count,
                SETTINGS.negative_count,
                SETTINGS.context_features,
                SETTINGS.update_mode,
            ],
        }
    )
)

## 1. Inspect the model contract in smaller groups

A feature is one numeric signal for an `(impression_uid, news_code)` pair. Only `schema["model_columns"]` may enter the ranker. All are stored as `float32`; a missing numeric value is represented by **zero plus a validity/missingness flag**, rather than assuming zero is a measured value.

The grouped tables below show the active feature names and their definitions. Any later scaler or model must be fitted on training rows only. Text/entity representations were prepared in part 3; no representation model is fitted in this notebook.

Features named `log_...` already use `log1p` ($\log(1 + x)$); do not apply that transform twice.

### 1.0 Edit features here

This notebook owns the feature catalogue, schema, calculations and sampling; `src/newsrec/features.py` is no longer needed.

1. **Choose groups:** edit `ENABLED_GROUPS` in the controls cell. Keep only the groups you want a ranker to use.
2. **Disable individual features:** add their exact names to `DISABLED_FEATURES`.
3. **Add a feature:** add an entry to `EXTRA_FEATURES` with a group, definition and `compute(frame, cfg)` calculation. The commented example is ready to uncomment. For a new group name, also add it to `ENABLED_GROUPS`.
4. Rerun from here, inspect the examples, then set `REBUILD=True` to regenerate the real feature dataset. Restore `REBUILD=False` afterward to reuse it.

The saved schema contains your selected model columns. Unselected built-in values can remain in the wide output for coverage, diagnostics or later ablations; they must not enter the model. Slate context additionally requires `SETTINGS.context_features=True`.

Changing selection/schema is detected when opening a cache. After editing a formula without changing its name/definition, explicitly set `REBUILD=True`. Use only static content or strictly earlier information when writing new calculations; missing values need a suitable flag.

In [ ]:
GROUPS = {
    "content": [  # What the article is: category, text length, entity coverage, embedding availability
        "category_code",          # Main category ID from the training vocabulary (0 = unseen)
        "subcategory_code",       # Subcategory ID from the training vocabulary (0 = unseen)
        "title_word_count",       # Number of words in the article title
        "abstract_word_count",    # Number of words in the article abstract
        "has_abstract",           # 1 if the article has a non-empty abstract, else 0
        "n_annotated_entities", # Number of annotated entities (100-dim article_entity = title entities + abstract entities) in the article
        "entity_coverage", # Fraction of annotated entities in the article that are present in the entity embedding vocabulary
        "article_has_text",       # 1 if the article has a usable text embedding, else 0
        "article_has_entity",     # 1 if the article has a usable entity embedding, else 0
    ],
    "history": [  # How much usable user history exists at time t, and which profiles could be built
        "log_history_length",             # log1p of the user's full causal history length
        "retained_history_length",        # Number of history items kept after the history_cap
        "valid_text_history_fraction",    # Fraction of retained history items with a valid text embedding
        "valid_entity_history_fraction",  # Fraction of retained history items with a valid entity embedding
        "no_supplied_history",            # 1 if the user had no history at all, else 0
        "no_causal_history",              # 1 if no history events are strictly before the request, else 0
        "no_text_profile",                # 1 if no text profile could be built, else 0
        "no_entity_profile",              # 1 if no entity profile could be built, else 0
        "no_recent_profile",              # 1 if no recent-window profile could be built, else 0
    ],
    "match": [  # How well the article matches the user's past interests (category counts + embedding cosines)
        "category_affinity",        # Prior clicked category count / full history count; zero if empty
        "subcategory_affinity",     # Prior clicked subcategory count / full history count; zero if empty
        "seen_category",            # 1 if the user's history contains the candidate's category, else 0
        "text_cosine",              # Cosine between article text embedding and long-term text profile
        "entity_cosine",            # Cosine between article entity embedding and long-term entity profile
        "long_term_cosine",         # Cosine vs the long-term profile (text or entity, per profile type)
        "recent_cosine",            # Cosine vs the recent-window profile (last 24h / 10 items)
        "text_match_valid",         # 1 if the text cosine is computed from valid vectors, else 0
        "entity_match_valid",       # 1 if the entity cosine is computed from valid vectors, else 0
        "long_term_match_valid",    # 1 if the long-term cosine is valid, else 0
        "recent_match_valid",       # 1 if the recent cosine is valid, else 0
    ],
    "repeat": [  # This user's prior interactions with this exact article (exposure/click fatigue)
        "log_prior_exposure_count",     # log1p of times this user saw this article before
        "log_prior_nonclick_count",     # log1p of prior exposures of this article that were not clicked
        "log_prior_pair_click_count",   # log1p of prior clicks on this exact (user, article) pair
        "hours_since_last_exposure",    # Hours since the user last saw this article; 0 + flag if never
        "no_prior_pair_exposure",       # 1 if the user never saw this article before, else 0
        "previously_clicked",           # 1 if the user clicked this article before, else 0
    ],
    "popularity": [  # How popular the article is across all users so far (smoothed CTR + support flags)
        "log_prior_article_exposures",   # log1p of the article's total prior exposures (all users)
        "log_prior_article_clicks",      # log1p of the article's total prior clicks (all users)
        "smoothed_prior_article_ctr",    # (clicks + strength * global CTR) / (exposures + strength), Beta(1,1) prior
        "article_low_support",           # 1 if the article has too few prior exposures to trust its CTR, else 0
        "population_startup",            # 1 if the article is new to the population (no prior exposure), else 0
    ],
    "freshness": [  # How long the article has been observed (age since first exposure, not publication)
        "article_age_hours",           # Hours since the article's first observed exposure; 0 + flag if none
        "no_prior_article_exposure",   # 1 if the article has never been exposed before, else 0
        "age_left_censored",           # 1 if the true age is unknown (left-censored at first observation), else 0
    ],
    "multi_interest": [  # best cosine vs up to 3 category-specific interest vectors
        "multi_interest_match_score",        # max(cos(article_title, I_1), cos(article_title, I_2), cos(article_title, I_3)) over up to three valid category interest vectors
        "interest_group_count",              # Number of valid interest vectors the user has
        "selected_interest_group_count",     # Number of interest vectors actually used for the match
        "multi_interest_valid",              # 1 if at least one valid interest vector existed, else 0
    ],
    "half_life": [  # Estimated content decay from early-vs-recent CTR, masked when unsupported
        "estimated_article_half_life_hours",  # ln(2) * 6h window-centre gap / ln(early CTR / recent CTR); 0 + mask if unsupported
        "has_half_life",                      # 1 if a half-life estimate was supported by enough data, else 0
        "stale_after_grace",                  # 1 if the article is older than its estimated half-life (past grace), else 0
    ],
    "request_context": [  # When the request happens: time of day, weekday, gap since last visit
        "day_part",                          # Time-of-day bucket of the request
        "weekday",                           # Day of week of the request
        "hours_since_previous_impression",   # Hours since the user's previous impression; 0 + flag if first
        "first_visit",                       # 1 if this is the user's first impression, else 0
    ],
    "slate_context": [  # Off for now. How the candidate compares to the other candidates shown in this same request (off by default)
        "personalized_relative_advantage",  # Candidate multi-interest score minus the median score of the candidates in this slate; 0 if invalid
        "slate_context_valid",              # 1 if every candidate in the slate had a valid multi-interest score, else 0
    ],
}

FORMULAS = {
    "category_affinity": "full prior clicked category count / full history count; zero if empty",
    "subcategory_affinity": "full prior clicked subcategory count / full history count; zero if empty",
    "seen_category": "prior candidate-category count > 0",
    "smoothed_prior_article_ctr": "(prior article clicks + strength * expanding Beta(1,1) global CTR) / (prior article exposures + strength)",
    "article_age_hours": "hours since first prior observed exposure; zero with flag if none",
    "estimated_article_half_life_hours": "ln(2) * completed 6h window-centre gap / ln(early CTR / recent CTR); zero plus mask when unsupported",
    "multi_interest_match_score": "max title64 cosine over up to three valid category interest vectors",
    "personalized_relative_advantage": "candidate multi-interest match minus actual candidate-set median; zero if context invalid",
}

In [ ]:
# Choose model inputs here. Removing a group excludes it from model_columns.
ENABLED_GROUPS = [
    "content",
    "history",
    "match",
    "repeat",
    "popularity",
    "freshness",
    "multi_interest",
    "half_life",
    "request_context",
    "slate_context",
    "custom",
]
# Example: DISABLED_FEATURES = ["weekday", "day_part"]
DISABLED_FEATURES = []

# Add a feature by registering its name, explanation and calculation in this cell.
# The frame contains already-computed causal features/diagnostics, never current labels.
EXTRA_FEATURES = {
    # "prior_click_share": {
    #     "group": "custom",
    #     "definition": "past article clicks / (past article exposures + 1)",
    #     "compute": lambda frame, cfg: frame["prior_article_clicks"]
    #                                    / (frame["prior_article_exposures"] + 1),
    # },
}

### Build the selected model contract

This function turns the editable catalogue and controls into the schema used by the examples, saved dataset, quality reports and later rankers. Shared validation rejects duplicate names, empty selections and protected key/target/diagnostic columns.

In [ ]:
def feature_schema(cfg=DEFAULT_SETTINGS):
    known_groups = (
        set(GROUPS)
        | {"custom"}
        | {s.get("group", "custom") for s in EXTRA_FEATURES.values()}
    )
    if set(ENABLED_GROUPS) - known_groups:
        raise ValueError(
            f"Unknown feature groups: {sorted(set(ENABLED_GROUPS) - known_groups)}"
        )
    builtin_names = {name for names in GROUPS.values() for name in names}
    if set(EXTRA_FEATURES) & builtin_names:
        raise ValueError("Custom feature names must not overwrite built-in features")
    if set(DISABLED_FEATURES) - (builtin_names | set(EXTRA_FEATURES)):
        raise ValueError("DISABLED_FEATURES contains an unknown feature name")
    features = []
    for group, names in GROUPS.items():
        if group not in ENABLED_GROUPS:
            continue
        if group == "slate_context" and (not cfg.context_features):
            continue
        for name in names:
            if name in DISABLED_FEATURES:
                continue
            features.append(
                {
                    "name": name,
                    "dtype": "float32",
                    "group": group,
                    "missing_value": 0,
                    "definition": FORMULAS.get(name, name.replace("_", " ")),
                    "cutoff": "static train-fitted content or events strictly before request timestamp",
                    "preprocessing": "log1p already applied"
                    if name.startswith("log_")
                    else "none; fit any scaler on training rows only",
                    "consumer": "ranker; slate context requires complete actual candidate set"
                    if group == "slate_context"
                    else "ranker",
                }
            )
    for name, specification in EXTRA_FEATURES.items():
        group = specification.get("group", "custom")
        if group not in ENABLED_GROUPS or name in DISABLED_FEATURES:
            continue
        if not callable(specification.get("compute")):
            raise ValueError(
                f"Custom feature {name} needs a compute(frame, cfg) function"
            )
        features.append(
            {
                "name": name,
                "dtype": "float32",
                "group": group,
                "missing_value": 0,
                "definition": specification["definition"],
                "cutoff": "static train-fitted content or events strictly before request timestamp",
                "preprocessing": "none; fit any scaler on training rows only",
                "consumer": "ranker",
            }
        )
    return validate_feature_schema(
        {
            "version": "ranking-features-v1",
            "features": features,
            "model_columns": [f["name"] for f in features],
            "keys": ["impression_uid", "news_code"],
            "diagnostics": [
                "user_code",
                "article_row",
                "timestamp",
                "state_version",
                "candidate_source",
                "eligible",
                "history_length",
                "prior_article_exposures",
                "prior_article_clicks",
                "prior_exposure_count",
                "half_life_reason",
                "half_life_early_support",
                "half_life_recent_support",
            ],
            "excluded": [
                "clicked",
                "position",
                "slate_len",
                "n_clicks",
                "n_negatives",
                "split",
                "user_code",
                "news_code",
                "impression_uid",
            ],
            "update_mode": cfg.update_mode,
            "config_hash": cfg.fingerprint(),
        }
    )

In [ ]:
schema = feature_schema(SETTINGS)
contract = pd.DataFrame(schema["features"])
assert not set(schema["model_columns"]) & set(schema["excluded"])
display(
    contract.groupby("group", sort=False).size().rename("active_features").to_frame()
)
print("Total active model features:", len(schema["model_columns"]))

### 1.1 Article content and representation coverage

What is this article, and do its representations exist? Category IDs and text lengths describe content; coverage flags tell a ranker whether a text/entity comparison can be trusted. Category encodings use the training vocabulary; an unseen category receives code 0.

In [ ]:
view = contract.loc[contract["group"].isin(["content"]), ["name", "definition"]]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))

### 1.2 User history and profile availability

How much causal history is available? Full-history length and retained-history length serve different purposes: counts can use all allowed earlier events, while the vector/sequence representation is capped. A zero cosine from a missing profile needs a separate flag.

In [ ]:
view = contract.loc[contract["group"].isin(["history"]), ["name", "definition"]]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))

### 1.3 User–article matching

How well does the candidate match the user? Normalised vectors make dot products cosine similarities. Category affinity uses earlier clicked-category counts. Long-term supplied history and recent timestamped clicks are separate views.

In [ ]:
view = contract.loc[contract["group"].isin(["match"]), ["name", "definition"]]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))

### 1.4 Repeated exposure

Has this user already seen or clicked this article? Exposure/non-click/click counts are taken before the request and log-transformed where named `log_`. The current impression contributes nothing until the batch is finished.

In [ ]:
view = contract.loc[contract["group"].isin(["repeat"]), ["name", "definition"]]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))

### 1.5 Popularity and observed freshness

Is the article popular so far, and how long has it been observed? The CTR feature is smoothed toward the expanding past-only global CTR. Age means time since first **prior observed exposure**, not publication time. A first exposure has age 0 with an explicit missing-prior flag.

In [ ]:
view = contract.loc[
    contract["group"].isin(["popularity", "freshness"]),
    ["name", "definition"],
]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))

### 1.6 Multiple interests and supported decay

A user can have several interests: the multi-interest score takes the maximum title-space cosine over valid category profiles. Half-life compares disjoint, completed six-hour windows, uses Beta(1,1) smoothed CTR, and requires support in both windows. Unsupported/no-decline estimates remain 0 with a mask; they are not evidence of an instant expiry.

In [ ]:
view = contract.loc[
    contract["group"].isin(["multi_interest", "half_life"]),
    ["name", "definition"],
]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))

### 1.7 Request context and optional slate context

Request context describes the clock and previous visit. Slate context is opt-in: `personalized_relative_advantage` subtracts the actual candidate-set median multi-interest score. Training uses the complete logged slate **before sampling**; serving must use the actual retrieved set. Changing the set can change this feature.

In [ ]:
view = contract.loc[
    contract["group"].isin(["request_context", "slate_context"]),
    ["name", "definition"],
]
if view.empty:
    print("No active features in this group.")
else:
    with pd.option_context("display.max_colwidth", None):
        display(view.reset_index(drop=True))
if not SETTINGS.context_features:
    print(
        "Slate context is disabled. Section 2.7 demonstrates it without changing the saved-build settings."
    )

### Your additional features

Registered custom features appear here when their group is enabled and their name is not disabled. Their values are calculated in `build_features` using the functions in `EXTRA_FEATURES`.

In [ ]:
custom_view = contract.loc[
    contract["name"].isin(EXTRA_FEATURES), ["name", "group", "definition"]
]
if custom_view.empty:
    print(
        "No custom features selected. Add an entry in EXTRA_FEATURES above to include one."
    )
else:
    display(custom_view.reset_index(drop=True))

### 1.8 Fields that must stay outside the model

Keep keys for joining predictions back to requests. Keep raw counts, state versions and half-life reasons for diagnostics. Keep `clicked` separately as a training/evaluation target. Shuffled slate position and current click/slate summaries are excluded from the model allowlist.

**Temporal rule:** score every request at timestamp `t`, then apply its allowed logged feedback. Rows sharing a train/validation boundary second are processed together. Prequential evaluation can update history from earlier validation/test events; frozen evaluation withholds feedback from its target split. Neither mode uses same-time or future feedback in the current feature vector.

In [ ]:
display(
    pd.DataFrame(
        {
            "purpose": [
                "Prediction join keys",
                "Diagnostics only",
                "Excluded from model inputs",
            ],
            "columns": [
                ", ".join(schema["keys"]),
                ", ".join(schema["diagnostics"]),
                ", ".join(schema["excluded"]),
            ],
        }
    )
)

## 2. Learn the methods through small worked examples

The following definitions drive the real build in section 3. Each definition is followed immediately by a small example. We start with pooling, then causal history, past-only freshness, article statistics, personalised matching, full feature assembly and negative sampling.

### 2.1 Pool valid history vectors

`normalised_mean` ignores missing/zero vectors, averages the remaining vectors and L2-normalises the mean. It also returns the number of usable history entries. In this example, two perpendicular vectors average to a direction equally close to both; the zero row contributes nothing.

In [ ]:
def normalised_mean(matrix, rows):
    if not len(rows):
        return (np.zeros(matrix.shape[1], dtype=np.float32), 0)
    selected = matrix[np.asarray(rows, dtype=np.int32)]
    valid = np.linalg.norm(selected, axis=1) > 1e-12
    if valid.any():
        pooled = normalise(selected[valid].mean(axis=0))
    else:
        pooled = np.zeros(matrix.shape[1], np.float32)
    return (pooled, int(valid.sum()))


# Synthetic example: row 0 is padding/missing; rows 1 and 2 are valid directions.
pooling_input = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]], dtype=np.float32)
pooled, valid_count = normalised_mean(pooling_input, [1, 2, 0])
display(
    pd.DataFrame(pooling_input, columns=["coordinate_1", "coordinate_2"]).rename_axis(
        "row"
    )
)
display(
    pd.DataFrame(
        [
            {
                "pooled_vector": np.round(pooled, 4).tolist(),
                "valid_history_entries": valid_count,
                "L2_norm": round(float(np.linalg.norm(pooled)), 4),
            }
        ]
    )
)
assert valid_count == 2
np.testing.assert_allclose(pooled, [1 / np.sqrt(2), 1 / np.sqrt(2)], atol=1e-6)

### A tiny request used by the next examples

Our synthetic user has supplied history `N101 N102`. The candidate slate contains sports article 101, science article 102 and mixed-content article 103. The user clicks 103 **in the current request**. We will show that this click is absent while scoring this request and present only at the next timestamp.

Vectors here are hand-made teaching coordinates, not embeddings learned from MIND. Article 103 deliberately has no entity vector, so its entity-match validity flag should be false. All three articles are available from the start of this toy stream.

In [ ]:
DEMO_SETTINGS = replace(
    SETTINGS, text_dim=4, entity_dim=3, history_cap=3, half_life_min_exposures=2
)
demo_start = pd.Timestamp("2019-11-09 00:00:00")
demo_end = demo_start + pd.Timedelta(hours=24)
demo_index = pd.DataFrame(
    {
        "news_code": [101, 102, 103],
        "article_row": [1, 2, 3],
        "category": ["sports", "science", "sports"],
        "subcategory": ["football", "space", "football"],
        "category_code": [1, 2, 1],
        "subcategory_code": [1, 2, 1],
        "title_word_count": [5, 4, 6],
        "abstract_word_count": [12, 10, 0],
        "has_abstract": [True, True, False],
    }
)
demo_lookup = dict(zip(demo_index.news_code, demo_index.article_row))
demo_text = np.zeros((4, DEMO_SETTINGS.text_dim), dtype=np.float32)
demo_text[1, 0] = demo_text[2, 1] = 1
demo_text[3, :2] = 1 / np.sqrt(2)
demo_entity = np.zeros((4, DEMO_SETTINGS.entity_dim), dtype=np.float32)
demo_entity[1, 0] = demo_entity[2, 1] = 1
demo_bundle = SimpleNamespace(
    n=4,
    index=demo_index,
    lookup=demo_lookup,
    text=demo_text,
    title=demo_text.copy(),
    entity=demo_entity,
    meta=demo_index.set_index("article_row"),
    quality=pd.DataFrame(
        {
            "news_code": [101, 102, 103],
            "n_annotated_entities": [1, 1, 0],
            "entity_coverage": [1.0, 1.0, 0.0],
        }
    ).set_index("news_code"),
    rows=lambda codes: np.asarray([demo_lookup[int(c)] for c in codes], dtype=np.int32),
    eligible=lambda codes, t: np.full(
        len(codes), pd.Timestamp(t) >= demo_start, dtype=bool
    ),
)
demo_requests = pd.DataFrame(
    {
        "impression_uid": [500],
        "user_code": [7],
        "timestamp": [demo_start],
        "history": ["N101 N102"],
    }
)
demo_events = pd.DataFrame(
    {
        "impression_uid": [500] * 3,
        "user_code": [7] * 3,
        "timestamp": [demo_start] * 3,
        "news_code": [101, 102, 103],
        "clicked": [0, 0, 1],
        "split": ["train"] * 3,
    }
)
display(demo_index[["news_code", "category", "has_abstract"]])
display(demo_requests)
display(demo_events[["impression_uid", "news_code", "clicked"]])

### 2.2 Causal profiles: begin → read profiles → finish

The class below inherits only the shared state allocation and pair-history storage. Its visible methods define the calculation:

- `begin` seeds supplied history once per user and checks the timestamp batch.
- `profile` returns capped history vectors, separate long-term/recent views, and category interest vectors. Supplied clicks have no timestamps, so the recent view uses logged clicks only.
- `finish` updates exposures, popularity and clicked history **after scoring**. Repeated genuine clicks are retained.

The example compares the same user before current feedback and at the next timestamp. Expect history length **2 → 3** and prior article-103 clicks **0 → 1**. When supplied-history order is unverified, the capped seed uses a deterministic order rather than claiming it is the most recent history.

In [ ]:
class CausalProfiles(_StateStorage):
    """Causal calculations; the base allocates state and the disk-backed pair store."""

    def begin(self, requests, t):
        t = pd.Timestamp(t)
        if self.active_time is not None:
            raise ValueError("Previous timestamp has not been finished")
        if self.last_update is not None and t <= self.last_update:
            raise ValueError(
                "Replay timestamps must increase strictly; merge same-time requests"
            )
        if t < self.start or t > self.end:
            raise ValueError("Request outside state horizon")
        if (
            requests.impression_uid.duplicated().any()
            or not requests.timestamp.eq(t).all()
        ):
            raise ValueError("begin requires unique requests from this timestamp only")
        self.active_time = t
        for row in requests.itertuples(index=False):
            text = "" if pd.isna(row.history) else str(row.history)
            codes = tuple(int(token.removeprefix("N")) for token in text.split())
            key = int(row.user_code)
            if key in self.users:
                if self.users[key].snapshot_key != codes:
                    raise ValueError(f"Supplied history changed for user {key}")
                continue
            snapshot = self.bundle.rows(codes)
            user = User(snapshot, codes, deque(maxlen=self.cfg.history_cap))
            user.categories.update(self.cat[snapshot])
            user.subcategories.update(self.subcat[snapshot])
            for article in snapshot:
                category = self.cat[article]
                if category not in user.interest_sums:
                    user.interest_sums[category] = np.zeros(self.cfg.text_dim, np.float32)
                user.interest_sums[category] += self.bundle.title[article]
            self.users[key] = user

    def profile(self, user_code, t):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError(
                "Call begin() for this timestamp before requesting profiles"
            )
        key = int(user_code)
        user = self.users[key]
        cache_key = (key, user.version, t.value)
        if cache_key in self.profiles:
            return self.profiles[cache_key]
        observed = list(user.observed)
        seed = (
            user.snapshot
            if self.cfg.snapshot_order_verified
            else np.sort(user.snapshot)
        )
        history = np.r_[seed, [a for _, a in observed]].astype(np.int32)[
            -self.cfg.history_cap :
        ]
        recent = [
            a for time, a in observed if t.value - time <= self.cfg.recent_hours * HOUR
        ][-self.cfg.recent_count :]
        text, valid_text = normalised_mean(self.bundle.text, history)
        entity, valid_entity = normalised_mean(self.bundle.entity, history)
        long, _ = normalised_mean(self.bundle.text, user.snapshot)
        recent_vector, valid_recent = normalised_mean(self.bundle.text, recent)
        interests = []
        interest_categories = []
        for cat, _ in sorted(user.categories.items(), key=lambda x: (-x[1], x[0]))[
            : self.cfg.interest_count
        ]:
            # Use all causal clicks, even when their rows leave the sequence cap.
            vector = user.interest_sums[cat]
            norm = np.linalg.norm(vector)
            if norm > 0:
                interests.append(vector / norm)
                interest_categories.append(cat)
        padded = np.zeros(self.cfg.history_cap, np.int32)
        padded[: len(history)] = history
        result = {
            "text": text,
            "entity": entity,
            "long_text": long,
            "recent_text": recent_vector,
            "interests": np.asarray(interests, np.float32).reshape(
                -1, self.cfg.text_dim
            ),
            "history_rows": padded,
            "history_mask": padded != 0,
            "selected_interest_categories": interest_categories,
            "recent_rows": list(map(int, recent)),
            "history_length": len(user.snapshot) + user.n_observed,
            "retained_length": len(history),
            "valid_text_fraction": valid_text / max(1, len(history)),
            "valid_entity_fraction": valid_entity / max(1, len(history)),
            "has_text": bool(np.linalg.norm(text) > 0),
            "has_entity": bool(np.linalg.norm(entity) > 0),
            "has_recent": bool(valid_recent and np.linalg.norm(recent_vector) > 0),
            "no_supplied_history": len(user.snapshot) == 0,
            "group_count": len(interests),
            "selected_group_count": min(self.cfg.interest_count, len(user.categories)),
            "state_version": user.version,
        }
        self.profiles[cache_key] = result
        return result

    def finish(self, requests, events, t, feedback_splits):
        t = pd.Timestamp(t)
        if self.active_time != t:
            raise ValueError("Timestamp is not active")
        if not events.timestamp.eq(t).all() or not set(events.impression_uid) <= set(
            requests.impression_uid
        ):
            raise ValueError("Feedback must belong to this timestamp's requests")
        owners = requests.set_index("impression_uid").user_code
        if not np.array_equal(
            events.user_code.to_numpy(), events.impression_uid.map(owners).to_numpy()
        ):
            raise ValueError("Feedback/request user mismatch")
        if events.duplicated(["impression_uid", "news_code"]).any():
            raise ValueError("Duplicate feedback event")
        for u in requests.user_code.unique():
            self.users[int(u)].last_impression = t
        selected = events[events.split.isin(feedback_splits)]
        if len(selected):
            if not selected.clicked.isin([0, 1]).all():
                raise ValueError("Feedback labels must be binary")
            rows = self.bundle.rows(selected.news_code)
            labels = selected.clicked.to_numpy(np.int64)
            self.pairs.update(selected.user_code, rows, labels, t)
            np.add.at(self.exposures, rows, 1)
            np.add.at(self.clicks, rows, labels)
            unseen = self.first[rows] < 0
            self.first[rows[unseen]] = t.value
            hour = int(t.value // HOUR - self.start_hour)
            np.add.at(self.hour_exposures[:, hour], rows, 1)
            np.add.at(self.hour_clicks[:, hour], rows, labels)
            self.total_exposures += len(rows)
            self.total_clicks += int(labels.sum())
            clicked = selected.loc[selected.clicked.eq(1)].sort_values(
                ["user_code", "news_code", "impression_uid"]
            )
            for event in clicked.itertuples(index=False):
                user = self.users[int(event.user_code)]
                a = self.bundle.lookup[int(event.news_code)]
                user.observed.append((t.value, a))
                user.n_observed += 1
                user.categories[self.cat[a]] += 1
                user.subcategories[self.subcat[a]] += 1
                category = self.cat[a]
                if category not in user.interest_sums:
                    user.interest_sums[category] = np.zeros(self.cfg.text_dim, np.float32)
                user.interest_sums[category] += self.bundle.title[a]
                user.version += 1
        self.profiles.clear()
        self.last_update = t
        self.active_time = None

### 2.3 Past-only half-life: evidence of declining CTR

`TemporalState` extends those profiles with a freshness estimator. It compares an early completed six-hour window against a later completed six-hour window. Both need sufficient exposure support; the recent CTR must be lower. The estimate is `ln(2) × window gap / ln(early CTR / recent CTR)`.

This is an observed-CTR decay estimate, not a publication lifetime or a hard retrieval expiry. A first observation exactly at the stream start remains left-censored. The example below deliberately supplies **synthetic earlier hourly aggregates**: 24/120 early clicks/exposures versus 12/120 recent. A second article has too little support and a third has no prior exposure. Changing counts in the current, unfinished hour must not change the estimate.

In [ ]:
class TemporalState(CausalProfiles):
    """Add a past-only freshness estimator to the causal profile state."""

    def half_life(self, rows, t):
        t = pd.Timestamp(t)
        recent_end = int(t.value // HOUR - self.start_hour)
        recent_start = recent_end - 6
        result = []
        for row in rows:
            first = self.first[row]
            if first < 0:
                result.append((0.0, False, "no_prior_exposure", 0, 0))
                continue
            early_start = int((first + HOUR - 1) // HOUR - self.start_hour)
            early_end = early_start + 6
            if recent_start < early_end:
                result.append((0.0, False, "young_or_overlapping", 0, 0))
                continue
            early_e = int(self.hour_exposures[row, early_start:early_end].sum())
            recent_e = int(self.hour_exposures[row, recent_start:recent_end].sum())
            if min(early_e, recent_e) < self.cfg.half_life_min_exposures:
                result.append((0.0, False, "low_support", early_e, recent_e))
                continue
            early_c = int(self.hour_clicks[row, early_start:early_end].sum())
            recent_c = int(self.hour_clicks[row, recent_start:recent_end].sum())
            early = (early_c + 1) / (early_e + 2)
            recent = (recent_c + 1) / (recent_e + 2)
            if recent >= early:
                result.append((0.0, False, "no_decline", early_e, recent_e))
                continue
            estimate = np.log(2) * (recent_start - early_start) / np.log(early / recent)
            ok = bool(np.isfinite(estimate) and estimate > 0)
            result.append(
                (
                    float(estimate) if ok else 0.0,
                    ok,
                    "available" if ok else "invalid",
                    early_e,
                    recent_e,
                )
            )
        return result


half_demo = TemporalState(demo_bundle, demo_start, demo_end, DEMO_SETTINGS)
try:
    half_demo.first[[1, 2]] = demo_start.value
    half_demo.hour_exposures[1, :6] = 20
    half_demo.hour_clicks[1, :6] = 4
    half_demo.hour_exposures[1, 12:18] = 20
    half_demo.hour_clicks[1, 12:18] = 2
    cutoff = demo_start + pd.Timedelta(hours=18)
    estimates = half_demo.half_life([1, 2, 3], cutoff)
    display(
        pd.DataFrame(
            estimates,
            columns=[
                "half_life_hours",
                "valid",
                "reason",
                "early_exposures",
                "recent_exposures",
            ],
        ).assign(news_code=[101, 102, 103])
    )
    half_demo.hour_exposures[1, 18] = 10000
    half_demo.hour_clicks[1, 18] = 9000
    assert half_demo.half_life([1], cutoff)[0] == estimates[0]
    print("Changing the unfinished hour leaves the estimate unchanged.")
finally:
    half_demo.close()

### 2.4 Article content, repeated exposure, popularity and age

`article_features` uses keyed pairs and the pre-feedback state to compute article-level signals. Its raw counts and reason strings remain diagnostics; only allowlisted fields enter the model.

The example uses the very first toy request. All prior exposure counts are zero; the expanding Beta(1,1) global CTR is 0.5, so startup smoothed article CTR is also 0.5 and `population_startup=True`. The current click on 103 cannot raise its popularity yet. Entity validity is false for 103 because we deliberately gave it a zero entity vector.

In [ ]:
def article_features(pairs, rows, state, bundle, cfg, t, candidate_source="retrieved"):
    """Static content and strictly prior pair/article statistics."""
    prior_pairs = state.pairs.lookup(pairs.user_code, rows)
    unique_rows, inverse = np.unique(rows, return_inverse=True)
    unique_half = state.half_life(unique_rows, t)
    half = [unique_half[i] for i in inverse]
    global_prior = (state.total_clicks + 1) / (state.total_exposures + 2)
    metadata = bundle.meta.loc[rows].reset_index(drop=True)
    quality = bundle.quality.loc[pairs.news_code.to_numpy()].reset_index(drop=True)
    article_e = state.exposures[rows]
    article_c = state.clicks[rows]
    first = state.first[rows]
    seen = first >= 0
    pair_e = np.asarray([x[0] for x in prior_pairs], np.int64)
    pair_c = np.asarray([x[1] for x in prior_pairs], np.int64)
    pair_last = np.asarray(
        [-1 if x[2] is None else x[2] for x in prior_pairs], np.int64
    )
    h = np.asarray([x[0] for x in half], np.float64)
    has_h = np.asarray([x[1] for x in half], bool)
    text_valid = np.linalg.norm(bundle.text[rows], axis=1) > 0
    entity_valid = np.linalg.norm(bundle.entity[rows], axis=1) > 0
    title_valid = np.linalg.norm(bundle.title[rows], axis=1) > 0
    ages = np.where(seen, (t.value - first) / HOUR, 0)
    # Raw counts stay diagnostic; log counts, CTR, age and masks feed the ranker.
    values = {
        "impression_uid": pairs.impression_uid.to_numpy(np.int64),
        "news_code": pairs.news_code.to_numpy(np.int64),
        "user_code": pairs.user_code.to_numpy(np.int64),
        "article_row": rows,
        "timestamp": t,
        "candidate_source": candidate_source,
        "eligible": True,
        "category_code": metadata.category_code.to_numpy(),
        "subcategory_code": metadata.subcategory_code.to_numpy(),
        "title_word_count": metadata.title_word_count.to_numpy(),
        "abstract_word_count": metadata.abstract_word_count.to_numpy(),
        "has_abstract": metadata.has_abstract.to_numpy(),
        "n_annotated_entities": quality.n_annotated_entities.to_numpy(),
        "entity_coverage": quality.entity_coverage.to_numpy(),
        "article_has_text": text_valid,
        "article_has_entity": entity_valid,
        "prior_exposure_count": pair_e,
        "log_prior_exposure_count": np.log1p(pair_e),
        "log_prior_nonclick_count": np.log1p(pair_e - pair_c),
        "log_prior_pair_click_count": np.log1p(pair_c),
        "hours_since_last_exposure": np.where(
            pair_last >= 0, (t.value - pair_last) / HOUR, 0
        ),
        "no_prior_pair_exposure": pair_last < 0,
        "prior_article_exposures": article_e,
        "prior_article_clicks": article_c,
        "log_prior_article_exposures": np.log1p(article_e),
        "log_prior_article_clicks": np.log1p(article_c),
        "smoothed_prior_article_ctr": (article_c + cfg.ctr_strength * global_prior)
        / (article_e + cfg.ctr_strength),
        "article_low_support": article_e < 100,
        "population_startup": state.total_exposures == 0,
        "article_age_hours": ages,
        "no_prior_article_exposure": ~seen,
        "age_left_censored": first == state.start.value,
        "estimated_article_half_life_hours": h,
        "has_half_life": has_h,
        "stale_after_grace": has_h & (ages > h + 6),
        "half_life_reason": [x[2] for x in half],
        "half_life_early_support": [x[3] for x in half],
        "half_life_recent_support": [x[4] for x in half],
        "day_part": t.hour // 6,
        "weekday": t.weekday(),
    }
    return values, {
        "text": text_valid,
        "entity": entity_valid,
        "title": title_valid,
        "prior_pair_clicks": pair_c,
    }


# Small reusable pre-feedback state for the next feature examples.
feature_demo = TemporalState(demo_bundle, demo_start, demo_end, DEMO_SETTINGS)
feature_demo.begin(demo_requests, demo_start)
demo_pairs, demo_rows, demo_time = prepare_feature_pairs(
    demo_requests, demo_events, feature_demo, demo_bundle, DEMO_SETTINGS, "logged"
)
article_values, demo_validity = article_features(
    demo_pairs, demo_rows, feature_demo, demo_bundle, DEMO_SETTINGS, demo_time, "logged"
)
display(
    pd.DataFrame(article_values)[
        [
            "news_code",
            "prior_article_clicks",
            "smoothed_prior_article_ctr",
            "population_startup",
            "article_age_hours",
            "no_prior_article_exposure",
            "article_has_entity",
        ]
    ]
)
assert not feature_demo.clicks.any()

### 2.5 Personalise the same candidates with the user’s history

`add_user_features` adds history coverage, category affinity, cosine matches and multi-interest scores. It reads profiles without updating them. A general profile averages interests; the multi-interest score takes the best category-specific match.

In this toy example the user’s history contains one sports and one science article. Both categories have affinity 0.5. Article 103 is equally aligned with the two directions, so it matches the pooled text profile well. Its entity cosine is zero **with a false validity flag**, not a measured entity disagreement. There are still no logged recent clicks at this first request.

Notice the distinction: article 103 can match the averaged profile best, while each single-topic article matches its own category interest best. The two features describe different views of the same user.

In [ ]:
def add_user_features(values, pairs, rows, state, bundle, cfg, t, validity):
    """Add profile matches without consuming any current labels or changing state."""
    n = len(pairs)
    pair_c = validity["prior_pair_clicks"]
    text_valid, entity_valid, title_valid = (
        validity[k] for k in ("text", "entity", "title")
    )
    user_columns = [
        "state_version",
        "history_length",
        "log_history_length",
        "retained_history_length",
        "valid_text_history_fraction",
        "valid_entity_history_fraction",
        "no_supplied_history",
        "no_causal_history",
        "no_text_profile",
        "no_entity_profile",
        "no_recent_profile",
        "category_affinity",
        "subcategory_affinity",
        "seen_category",
        "text_cosine",
        "entity_cosine",
        "long_term_cosine",
        "recent_cosine",
        "previously_clicked",
        "multi_interest_match_score",
        "interest_group_count",
        "selected_interest_group_count",
        "multi_interest_valid",
        "hours_since_previous_impression",
        "first_visit",
        "text_match_valid",
        "entity_match_valid",
        "long_term_match_valid",
        "recent_match_valid",
    ]
    values.update({name: np.zeros(n, np.float64) for name in user_columns})
    for user_code, positions in pairs.groupby("user_code", sort=False).indices.items():
        idx = np.asarray(positions)
        article_rows = rows[idx]
        user = state.users[int(user_code)]
        p = state.profile(user_code, t)
        total = p["history_length"]
        category_counts = np.asarray(
            [user.categories[c] for c in state.cat[article_rows]]
        )
        subcategory_counts = np.asarray(
            [user.subcategories[c] for c in state.subcat[article_rows]]
        )
        dynamic = {
            "state_version": p["state_version"],
            "history_length": total,
            "log_history_length": np.log1p(total),
            "retained_history_length": p["retained_length"],
            "valid_text_history_fraction": p["valid_text_fraction"],
            "valid_entity_history_fraction": p["valid_entity_fraction"],
            "no_supplied_history": p["no_supplied_history"],
            "no_causal_history": total == 0,
            "no_text_profile": not p["has_text"],
            "no_entity_profile": not p["has_entity"],
            "no_recent_profile": not p["has_recent"],
            "category_affinity": category_counts / max(1, total),
            "subcategory_affinity": subcategory_counts / max(1, total),
            "seen_category": category_counts > 0,
            "text_cosine": bundle.text[article_rows] @ p["text"],
            "entity_cosine": bundle.entity[article_rows] @ p["entity"],
            "long_term_cosine": bundle.text[article_rows] @ p["long_text"],
            "recent_cosine": bundle.text[article_rows] @ p["recent_text"],
            "text_match_valid": p["has_text"] & text_valid[idx],
            "entity_match_valid": p["has_entity"] & entity_valid[idx],
            "long_term_match_valid": bool(np.linalg.norm(p["long_text"]) > 0)
            & text_valid[idx],
            "recent_match_valid": p["has_recent"] & text_valid[idx],
            "previously_clicked": (pair_c[idx] > 0)
            | np.isin(article_rows, user.snapshot),
            "multi_interest_match_score": np.max(
                bundle.title[article_rows] @ p["interests"].T, axis=1
            )
            if p["group_count"]
            else np.zeros(len(idx)),
            "interest_group_count": p["group_count"],
            "selected_interest_group_count": p["selected_group_count"],
            "multi_interest_valid": bool(p["group_count"]) & title_valid[idx],
            "hours_since_previous_impression": (
                t - user.last_impression
            ).total_seconds()
            / 3600
            if user.last_impression is not None
            else 0,
            "first_visit": user.last_impression is None,
        }
        for name, value in dynamic.items():
            values[name][idx] = value
    return values


personalised_values = add_user_features(
    article_values.copy(),
    demo_pairs,
    demo_rows,
    feature_demo,
    demo_bundle,
    DEMO_SETTINGS,
    demo_time,
    demo_validity,
)
display(
    pd.DataFrame(personalised_values)[
        [
            "news_code",
            "history_length",
            "category_affinity",
            "text_cosine",
            "entity_cosine",
            "entity_match_valid",
            "recent_match_valid",
            "multi_interest_match_score",
        ]
    ]
)
assert feature_demo.users[7].n_observed == 0

### 2.6 Assemble a model-ready feature table

`build_features` joins the previous two calculations, applies optional candidate-set context and casts the model columns. The shared input helper enforces unique keys, a single active timestamp and article eligibility.

The function consumes no labels and does not update state. Below, complete logged candidates and the same candidates with labels removed must produce identical model features. The displayed keys help join predictions back later; they are not model inputs.

In [ ]:
def build_features(
    requests,
    candidate_pairs,
    state_before_t,
    representation_bundle,
    feature_settings=DEFAULT_SETTINGS,
    candidate_source="retrieved",
):
    """Same input contract for logged or unlabelled retrieved candidates.

    Caller begins one complete timestamp, builds all feature requests, then calls
    finish with logged feedback. This function neither reads labels nor mutates state.
    """
    state = state_before_t
    bundle = representation_bundle
    cfg = feature_settings
    pairs, rows, t = prepare_feature_pairs(
        requests, candidate_pairs, state, bundle, cfg, candidate_source
    )
    values, validity = article_features(
        pairs, rows, state, bundle, cfg, t, candidate_source
    )
    values = add_user_features(values, pairs, rows, state, bundle, cfg, t, validity)
    result = pd.DataFrame(values)
    result["state_version"] = result.state_version.astype(np.int64)
    result["history_length"] = result.history_length.astype(np.int64)
    if cfg.context_features:
        grouped = result.groupby("impression_uid")
        valid = grouped.multi_interest_valid.transform("all")
        result["slate_context_valid"] = valid
        result["personalized_relative_advantage"] = (
            result.multi_interest_match_score
            - grouped.multi_interest_match_score.transform("median")
        ).where(valid, 0)
    active_contract = feature_schema(cfg)
    columns = active_contract["model_columns"]
    for name, specification in EXTRA_FEATURES.items():
        if name in columns:
            result[name] = specification["compute"](result, cfg)
    result[columns] = result[columns].astype(np.float32)
    if not np.isfinite(result[columns].to_numpy()).all():
        raise ValueError("Nonfinite model features")
    return result


try:
    demo_features = build_features(
        demo_requests, demo_events, feature_demo, demo_bundle, DEMO_SETTINGS, "logged"
    )
    demo_without_labels = build_features(
        demo_requests,
        demo_events[["impression_uid", "news_code"]],
        feature_demo,
        demo_bundle,
        DEMO_SETTINGS,
        "retrieved",
    )
    pd.testing.assert_frame_equal(
        demo_features[schema["model_columns"]],
        demo_without_labels[schema["model_columns"]],
    )
    display(
        demo_features[
            [
                "impression_uid",
                "news_code",
                "text_cosine",
                "log_prior_article_clicks",
                "article_age_hours",
                "no_recent_profile",
            ]
        ]
    )
    print(
        "Same model features with or without current labels; state is still unchanged."
    )
    assert feature_demo.total_exposures == 0
finally:
    feature_demo.close()

### 2.7 Optional slate context in action

This example temporarily enables context on the toy slate only; it does not change `SETTINGS` for the real build. Relative advantage is the candidate’s multi-interest score minus the slate median. Positive means stronger interest matching than the typical candidate in **this set**.

All candidates must have valid multi-interest matches for this context to be considered valid. At inference, removing or adding candidates can change the median; never reuse a median computed from a different slate.

In [ ]:
context_settings = replace(DEMO_SETTINGS, context_features=True)
context_demo = TemporalState(demo_bundle, demo_start, demo_end, context_settings)
try:
    context_demo.begin(demo_requests, demo_start)
    context_features = build_features(
        demo_requests,
        demo_events[["impression_uid", "news_code"]],
        context_demo,
        demo_bundle,
        context_settings,
    )
    display(
        context_features[
            [
                "news_code",
                "multi_interest_match_score",
                "personalized_relative_advantage",
                "slate_context_valid",
            ]
        ]
    )
    np.testing.assert_allclose(
        context_features.personalized_relative_advantage,
        context_features.multi_interest_match_score
        - context_features.multi_interest_match_score.median(),
        atol=1e-6,
    )
finally:
    context_demo.close()

### 2.8 Sample exposed negatives for training

`sample_training_groups` makes a stable seeded draw for each positive article in a logged training impression. It samples exposed non-clicks without replacement within that group. A negative can belong to multiple positive groups; the pointwise table keeps the union of unique pairs.

The tiny example uses **two negatives per positive for readability**; the production default is four. Shuffling input rows must not change the sampled groups. Sampling only selects written training examples: causal state still consumes the complete earlier logged stream. Validation/test keep their complete candidate slates.

In [ ]:
def sample_training_groups(candidates, cfg=DEFAULT_SETTINGS):
    """Reproducible per-positive exposed negatives, independent of input row order."""
    groups = []
    selected = set()
    for uid, slate in candidates.groupby("impression_uid", sort=True):
        if slate.news_code.duplicated().any() or not slate.clicked.isin([0, 1]).all():
            raise ValueError("Sampling requires unique binary-labelled logged pairs")
        positives = sorted(map(int, slate.loc[slate.clicked.eq(1), "news_code"]))
        negatives = np.sort(
            slate.loc[slate.clicked.eq(0), "news_code"].to_numpy(np.int64)
        )
        if not positives:
            raise ValueError("Logged training impression has no positives")
        for positive in positives:
            seed = int.from_bytes(
                hashlib.sha256(f"{cfg.seed}:{int(uid)}:{positive}".encode()).digest()[
                    :8
                ],
                "little",
            )
            rng = np.random.default_rng(seed)
            sampled = sorted(
                map(
                    int,
                    rng.choice(
                        negatives,
                        min(cfg.negative_count, len(negatives)),
                        replace=False,
                    ),
                )
            )
            groups.append(
                {
                    "impression_uid": int(uid),
                    "positive_news_code": positive,
                    "negative_news_codes": sampled,
                }
            )
            selected.add((int(uid), positive))
            selected.update((int(uid), n) for n in sampled)
    return (
        pd.DataFrame(
            groups,
            columns=["impression_uid", "positive_news_code", "negative_news_codes"],
        ),
        selected,
    )


sampling_input = pd.DataFrame(
    {
        "impression_uid": [700] * 8,
        "news_code": list(range(201, 209)),
        "clicked": [1, 0, 0, 1, 0, 0, 0, 0],
    }
)
sampling_settings = replace(DEMO_SETTINGS, negative_count=2)
demo_groups, selected_pairs = sample_training_groups(sampling_input, sampling_settings)
shuffled_groups, shuffled_pairs = sample_training_groups(
    sampling_input.sample(frac=1, random_state=1), sampling_settings
)
pd.testing.assert_frame_equal(demo_groups, shuffled_groups)
assert selected_pairs == shuffled_pairs
display(sampling_input)
display(demo_groups)
display(pd.DataFrame(sorted(selected_pairs), columns=["impression_uid", "news_code"]))
print("Groups are reproducible after shuffling the input.")

### 2.9 Add your own feature using the same registry format

This isolated example shows the exact format used by `EXTRA_FEATURES`: a group, a definition and a `compute(frame, cfg)` function. The input counts are deliberately synthetic and refer to earlier events only. Adding one to the denominator avoids division by zero; it does not turn the number into a calibrated CTR.

To use this feature in the real build, uncomment the matching entry in the controls cell near the top (or copy this entry there), keep `"custom"` enabled, and rebuild. This example itself does not change your selected production features.

In [ ]:
custom_example = {
    "group": "custom",
    "definition": "past article clicks / (past article exposures + 1)",
    "compute": lambda frame, cfg: (
        frame["prior_article_clicks"] / (frame["prior_article_exposures"] + 1)
    ),
}
custom_input = pd.DataFrame(
    {
        "news_code": [101, 102, 103],
        "prior_article_clicks": [2, 0, 1],
        "prior_article_exposures": [10, 0, 3],
    }
)
custom_preview = custom_input.copy()
custom_preview["prior_click_share"] = custom_example["compute"](
    custom_input, DEMO_SETTINGS
)
display(custom_preview)
print(
    "Copy the prior_click_share entry into EXTRA_FEATURES to add it to the saved schema."
)

## 3. Build or reopen the actual MIND feature dataset

The examples above were synthetic and in memory. This cell uses your prepared inputs, real representation bundle and the **same functions/classes defined above**.

- The terminal runner defaults to a full build. Opening this notebook directly defaults to the first 1,000 chronological timestamps for a smoke check; `MAX_TIMESTAMPS=None` selects the full replay.
- Prequential mode updates state only after each timestamp. For frozen evaluation choose a target split and `Settings(update_mode="frozen")`; earlier splits still supply history.
- `REBUILD=False` loads a complete, verified compatible run. `REBUILD=True` replaces the selected run directory.

The build uses bounded Parquet batches and disk-backed pair history. A smoke result is not a full evaluation; it can legitimately have no validation/test partitions. The row counts printed below are written feature rows **after training sampling**.

In [ ]:
# A notebook run defaults to a small smoke build; the terminal runner defaults to full.
MAX_TIMESTAMPS = (
    None
    if os.environ.get("NEWSREC_FULL_FEATURES") == "1"
    else int(os.environ.get("NEWSREC_SMOKE_TIMESTAMPS", "1000"))
)
TARGET_SPLIT = None  # Set validation/test with Settings(update_mode='frozen').
OVERWRITE = REBUILD or os.environ.get("NEWSREC_OVERWRITE_FEATURES") == "1"
run_dir = feature_run(
    ROOT,
    SETTINGS,
    target_split=TARGET_SPLIT,
    max_timestamps=MAX_TIMESTAMPS,
    rebuild=OVERWRITE,
    state_factory=TemporalState,
    feature_builder=build_features,
    negative_sampler=sample_training_groups,
    schema_builder=feature_schema,
)
print(run_dir)

run_manifest = json.loads((run_dir / "run_manifest.json").read_text())
display(
    pd.DataFrame(
        {
            "split": list(run_manifest["rows_written"]),
            "written_feature_rows": list(run_manifest["rows_written"].values()),
        }
    )
)

## 4. Audit the real saved outputs

The remaining tables come from `run_dir`, not the toy examples. Coverage and feature-quality reports describe all scored logged pairs **before training sampling**, so their denominators can exceed the written training-row count.

### 4.1 Representation and profile coverage

Start with all shown exposures. A low valid fraction indicates how often the ranker will rely on a missing-vector fallback. Article-vector coverage and user-profile coverage are different quantities. The complete CSV also contains clicked/new-user/new-article cohorts for later cold-start analysis.

In [ ]:
coverage = pd.read_csv(run_dir / "coverage.csv")
display(
    coverage.loc[
        coverage.cohort.eq("shown_exposures"),
        [
            "split",
            "representation",
            "denominator",
            "valid",
            "fallback",
            "valid_fraction",
        ],
    ].reset_index(drop=True)
)
print("All saved coverage cohorts:", ", ".join(sorted(coverage.cohort.unique())))

### 4.2 Value ranges, missingness and numerical validity

The summary below helps identify broken values without dumping the full feature table. A high zero fraction can be expected for sparse behaviour or missing profiles; read it together with validity flags. Finite/range checks do not prove ranking quality or remove the need for downstream ablations.

The first table averages zero fractions within each feature group; it is a numerical summary, not a representation fallback rate.

In [ ]:
quality = pd.read_csv(run_dir / "feature_quality.csv")
quality_by_group = quality.merge(
    contract[["name", "group"]],
    left_on="feature",
    right_on="name",
    how="left",
    validate="many_to_one",
)
zero_summary = (
    quality_by_group.groupby(["group", "split"])["zero_fraction"]
    .mean()
    .unstack("split")
)
print("Mean zero fraction by group and split:")
display(zero_summary.round(3))
print("Total nonfinite values:", int(quality.nonfinite_count.sum()))
display(
    quality.sort_values("zero_fraction", ascending=False)[
        ["split", "feature", "min", "max", "zero_fraction"]
    ].head(8)
)
assert quality.nonfinite_count.sum() == 0

### 4.3 Check what training sampling retained

Positive-group memberships and unique written negative pairs are different counts. A negative reused across two groups contributes two memberships but only one pointwise row. The discarded-negative fraction describes the reduction in written training negatives, not feedback discarded from history.

In [ ]:
sampling_summary = json.loads((run_dir / "sampling_summary.json").read_text())
display(pd.DataFrame(sampling_summary.items(), columns=["sampling_measure", "value"]))

### 4.4 Inspect one real request–article row

Read a small Arrow batch instead of loading the full feature dataset into pandas. The first table includes join keys and selected signals. The second annotates the first candidate’s model values using the same feature groups from section 1. Which missingness flags explain a zero value?

In [ ]:
real_sample = feature_sample(run_dir, split="train", rows=3)
if real_sample.empty:
    print("This run contains no written training partition.")
else:
    display(
        real_sample[
            [
                "impression_uid",
                "news_code",
                "history_length",
                "text_cosine",
                "multi_interest_match_score",
                "article_age_hours",
                "no_text_profile",
                "has_half_life",
            ]
        ]
    )
    first_candidate = contract[["group", "name"]].copy()
    first_candidate["value"] = [
        real_sample.iloc[0][name] for name in first_candidate.name
    ]
    display(
        first_candidate.loc[
            (
                first_candidate.group.isin(["history", "match", "freshness"])
                | first_candidate.name.isin(EXTRA_FEATURES)
            ),
            :,
        ].reset_index(drop=True)
    )

### 4.5 Runtime and resource cost

These are measurements recorded when this selected dataset was built. Loading cached outputs today does not rerun the historical build or replace its recorded runtime. Peak process RSS is sampled process memory, not total system memory; artifact size includes the saved feature-run files.

In [ ]:
cost = pd.read_csv(run_dir / "feature_cost.csv")
display(
    cost[
        [
            "run",
            "wall_seconds",
            "sampled_peak_process_rss_mb",
            "artifact_bytes",
            "emitted_rows",
            "timestamp_batches",
        ]
    ]
)

## 5. Verify inference parity on real inputs

Candidate generation later supplies only `(impression_uid, news_code)` pairs from the eligible catalogue. Recompute features with labels removed for the same first real timestamp. The model columns must agree; then consume feedback after scoring.

This check covers the same candidate set. Optional slate-context features may change when retrieval supplies a different set. The small table is a real-data example, separate from the toy checks in section 2.

In [ ]:
# Execute the same visible functions on one complete timestamp, before feedback.
bundle = Bundle(ROOT, SETTINGS)
state = TemporalState(bundle, *time_horizon(ROOT), SETTINGS)
try:
    t, requests, logged = next(replay_batches(ROOT, SETTINGS))
    state.begin(requests, t)
    scored = build_features(requests, logged, state, bundle, SETTINGS, "logged")
    unlabelled = build_features(
        requests,
        logged[["impression_uid", "news_code"]],
        state,
        bundle,
        SETTINGS,
        "retrieved",
    )
    pd.testing.assert_frame_equal(
        scored[schema["model_columns"]], unlabelled[schema["model_columns"]]
    )
    display(
        scored[
            [
                "impression_uid",
                "news_code",
                "text_cosine",
                "prior_article_clicks",
                "history_length",
            ]
        ].head(3)
    )
    print("Compared model columns:", len(schema["model_columns"]))
    # All requests at t have been scored before feedback at t is consumed.
    feedback_splits = {"train", "validation", "test"}
    if SETTINGS.update_mode == "frozen":
        feedback_splits.discard(TARGET_SPLIT)
    state.finish(requests, logged, t, feedback_splits)
    print("Logged/retrieved parity passed; feedback applied only after scoring.")
finally:
    state.close()

## 6. Handoff to baselines, retrieval and re-ranking

| Saved output | How later work uses it |
|---|---|
| `feature_schema.json` | Select only `model_columns`; keep keys/diagnostics out of the model matrix. |
| `ranking_features_<split>/` | Read feature partitions by split and selected columns. |
| `labels_<split>/` | Join `clicked` by `(impression_uid, news_code)` for supervised training/evaluation. |
| `request_profiles_<split>/` | Reconstruct the exact causal history, masks and interest references for a request. |
| `training_negative_groups/` | Preserve positive-to-negative memberships for grouped losses. |
| Coverage, quality, sampling and cost reports | Report availability, numerical validity, sampling choices and computational cost. |
| `run_manifest.json`, `run_status.json` | Confirm completion, settings/dependencies, smoke/full scope and evaluation mode. |

For unlabelled retrieved candidates, reuse `build_features` with the same pre-feedback request state and representation settings. The function does not invent labels for unshown articles. Fit any downstream scaler, clusters or ranker using training data only.

**Limits to state in the report:** observed age is not publication age; unsupported decay is masked; the offline representation space is fitted on the training catalogue; evaluation must specify prequential versus frozen feedback. Runtime/coverage checks alone do not establish a ranking improvement or CTR uplift. Graph embeddings, interest clustering, ANN retrieval and learned rankers belong to later stages.

If you change the causal profile calculation, keep its repeated methods consistent with `3_03_user_profiles.ipynb`, then rebuild affected outputs. No rebuild is needed merely for editing explanations or displaying additional examples.